# VideoDB QA Preview

<a href="https://colab.research.google.com/github/video-db/videodb-cookbook/blob/preview/guides/preview/video_qa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Ask questions about a video with VideoDB's preview **QA sessions** API.

The high-level SDK wrapper for QA is not added yet, so QA calls use `conn.post()` and `conn.get()` directly. `conn.post()` automatically waits for VideoDB async responses.

**Notebook owners:** Pradhan and KKS.

## 1. Install and Connect

In [ ]:
!pip install -q "git+https://github.com/Video-DB/videodb-python.git@indexing-v2" python-dotenv

In [ ]:
import os
import re
from getpass import getpass

from videodb import SceneExtractionType, connect

if not os.environ.get("VIDEO_DB_API_KEY"):
    os.environ["VIDEO_DB_API_KEY"] = getpass("Enter your VideoDB API key: ")

conn = connect(api_key=os.environ["VIDEO_DB_API_KEY"])
coll = conn.get_collection()

## 2. Upload a Video

You can replace the URL with any public video URL, or swap this cell for `coll.get_video("m-...")`.

In [ ]:
video = coll.upload("https://www.youtube.com/watch?v=vVlEVRKv4is")
video.play()

print("Video ID:", video.id)

## 3. Build Indexes

QA works best when both spoken-word and scene indexes are complete.

This cell creates/reuses both:
- **Spoken-word index** for transcript and dialogue questions
- **Scene index** for visual and timestamped scene questions

In [ ]:
# Spoken-word index
video.index_spoken_words(force=True)
print("✅ Spoken-word index ready")

# Scene index
try:
    scene_index_id = video.index_scenes(
        extraction_type=SceneExtractionType.shot_based,
        prompt="Describe the visual content in this scene for question answering.",
    )
except Exception as e:
    # If a scene index already exists, the SDK error usually includes its ID.
    match = re.search(r"id\s+([a-f0-9]+)", str(e))
    if not match:
        raise
    scene_index_id = match.group(1)

print("✅ Scene index ready:", scene_index_id)

## 4. Ask Your First Question

Start a QA session with `POST /qa/?video_id=<video_id>`.

The response includes a `session_id`; keep it for follow-up questions.

In [ ]:
qa_result = conn.post(
    f"qa/?video_id={video.id}",
    data={
        "query": "What happens in this video? Give me a concise summary and cite relevant moments if possible.",
        "model_name": "basic",
        "response_mode": "auto",
    },
    show_progress=True,
)

session_id = qa_result["session_id"]
print("Session ID:", session_id)
qa_result

## 5. Read the Response

QA returns an ordered list of response blocks. Common block types are:

- `text` — answer text
- `clips` — timestamped video clips
- `references` — evidence used by the agent
- `videos` — related video references

In [ ]:
for block in qa_result.get("response", []):
    block_type = block.get("type")
    print(f"\n--- {block_type} ---")

    if block_type == "text":
        print(block.get("text"))

    elif block_type == "clips":
        for clip in block.get("clips", []):
            print(f"{clip['video_id']}: {clip['start']}s → {clip['end']}s")
            if clip.get("reason"):
                print("Reason:", clip["reason"])

    elif block_type == "references":
        for ref in block.get("references", []):
            print(ref)

    else:
        print(block)

## 6. Ask a Follow-up

Continue the same chat with `POST /qa/<session_id>/`.

The server loads the previous conversation state automatically.

In [ ]:
followup_result = conn.post(
    f"qa/{session_id}/",
    data={
        "query": "Which part should I watch if I only have 30 seconds? Return clips if you are confident.",
        "model_name": "basic",
        "response_mode": "clips",
    },
    show_progress=True,
)

followup_result

## 7. List and Inspect Sessions

In [ ]:
sessions = conn.get(f"qa/?video_id={video.id}")
sessions

In [ ]:
session_details = conn.get(f"qa/{session_id}/?include_messages=false")
session_details

## Next Steps

Try questions like:

- “What are the key moments in this video?”
- “Where does the main action happen?”
- “Find clips where someone is speaking.”
- “Summarize the visual timeline.”